# Global test

## 1. Mkini from 49 to 143

### 1.1 Sqeuential mkini

In [7]:
import subprocess
import os
from joblib import Parallel, delayed
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed  # 切换到 ProcessPoolExecutor 以支持实时进度
import time
import sys

def load_environment(env_file):
    cmd = f'bash -c "source {env_file} && env"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    new_env = {}
    for line in result.stdout.strip().split('\n'):
        if '=' in line:
            key, value = line.split('=', 1)
            if not key.startswith('BASH_FUNC_'):
                new_env[key] = value
    
    os.environ.update(new_env)
    return os.environ.copy()


if __name__ == "__main__":
    env_file = '/share/home/dq089/soft/gnu-env'
    run_path = '/share/home/dq076/mode/ME/CoLM202X_CH4_g/run/'
    lsf_path = f'{run_path}g1.lsf'
    log_path = f'{run_path}logs/'  
    os.makedirs(log_path, exist_ok=True)

    updated_env = load_environment(env_file)

    for n in range(49,144):
        log_file = f'{log_path}g{n}-o.out'

        if os.path.exists(log_file):
            print(f"日志文件 {log_file} 已存在，跳过作业 g{n}。")
            continue

        with open(lsf_path, 'r') as file:
            lsf = file.readlines()

        for i, line in enumerate(lsf):
            if '#BSUB -J g' in line:
                lsf[i] = f'#BSUB -J g{n}\n'
            if '#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
                lsf[i] = f'#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-o.out\n'
            if '#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
                lsf[i] = f'#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-e.out\n'
            if '#BSUB -n' in line:
                lsf[i] = f'#BSUB -n {n}\n'
            if '/share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n' in line:
                lsf[i] = f'mpirun -np {n} /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n'
        
        with open(lsf_path, 'w') as file:
            file.writelines(lsf)
        
        sub= f'bsub<{lsf_path}'
        os.system(sub)
        # subprocess.run(sub, 
        #                 env=updated_env, 
        #                 stderr=subprocess.STDOUT, 
        #                 text=True)
        
        print(f"提交作业 g{n}，等待日志文件 {log_file} 生成...")
        while not os.path.exists(log_file):
            time.sleep(1)  # 每秒检查一次

        # 文件存在后，检查内容
        with open(log_file, 'r', encoding='utf-8') as f:
            content = f.read()

        if "CoLM Initialization Execution Completed" in content:
            print(f"日志文件 {log_file} 已生成且包含完成消息，继续下一个作业。")
        else:
            print(f"日志文件 {log_file} 已生成，但未找到 'CoLM Initialization Execution Completed'，程序结束。")
            sys.exit(1)  # 或 raise ValueError("作业未完成") 以结束程序

Job <115319> is submitted to queue <normal>.
提交作业 g49，等待日志文件 /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g49-o.out 生成...


KeyboardInterrupt: 

### 1.2 Dichotomy to mkini

In [10]:
import os
import time
import sys

env_file = '/share/home/dq089/soft/gnu-env'
run_path = '/share/home/dq076/mode/ME/CoLM202X_CH4_g/run/'
lsf_path = f'{run_path}g1.lsf'
log_path = f'{run_path}logs/'  
os.makedirs(log_path, exist_ok=True)

def load_environment(env_file):
    cmd = f'bash -c "source {env_file} && env"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    new_env = {}
    for line in result.stdout.strip().split('\n'):
        if '=' in line:
            key, value = line.split('=', 1)
            if not key.startswith('BASH_FUNC_'):
                new_env[key] = value
    
    os.environ.update(new_env)
    return os.environ.copy()

def test_n(n):
    """
    測試單個 n 是否成功：修改 LSF、提交、等待、檢查。
    返回 False 如果失敗（無完成消息），True 如果成功。
    """
    log_file = f'{log_path}g{n}-o.out'
    if os.path.exists(log_file):
        print(f"日志文件 {log_file} 已存在，跳过作业 g{n}。")
        with open(log_file, 'r', encoding='utf-8') as f:
            content = f.read()
        return "CoLM Initialization Execution Completed" in content
    
    # 不存在：執行提交流程
    with open(lsf_path, 'r') as file:
        lsf = file.readlines()

    for i, line in enumerate(lsf):
        if '#BSUB -J g' in line:
            lsf[i] = f'#BSUB -J g{n}\n'
        if '#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
            lsf[i] = f'#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-o.out\n'
        if '#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
            lsf[i] = f'#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-e.out\n'
        if '#BSUB -n' in line:
            lsf[i] = f'#BSUB -n {n}\n'
        if '/share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n' in line:
            lsf[i] = f'mpirun -np {n} /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n'
    
    
    with open(lsf_path, 'w') as file:
        file.writelines(lsf)
    
    sub = f'bsub<{lsf_path}'
    os.system(sub)
    
    print(f"测试 n={n}，等待日志...")
    while not os.path.exists(log_file):
        time.sleep(1)
    
    with open(log_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    success = "CoLM Initialization Execution Completed" in content
    if not success:
        print(f"n={n} 失败。")
    return success

# 二分搜索：找第一個失敗 n（假設單調）
def find_first_failing_n(low=49, high=143):
    while low < high:
        mid = (low + high) // 2
        if not test_n(mid):  # 如果 mid 失敗，邊界在低半邊
            high = mid
        else:  # 成功，邊界在高半邊
            low = mid + 1
    # 驗證 low 是否失敗
    if test_n(low):
        return None  # 無失敗
    return low

if __name__ == "__main__":
    updated_env = load_environment(env_file)

    first_failing = find_first_failing_n()
    if first_failing:
        print(f"第一個失敗 n: {first_failing}")
        print(f"預計所有 >= {first_failing} 均失敗。")
        # 可選：線性驗證後續
    else:
        print("所有 n 均成功！")

Job <115380> is submitted to queue <normal>.
测试 n=96，等待日志...
Job <115381> is submitted to queue <normal>.
测试 n=120，等待日志...
n=120 失败。
Job <115382> is submitted to queue <normal>.
测试 n=108，等待日志...
n=108 失败。
Job <115383> is submitted to queue <normal>.
测试 n=102，等待日志...
Job <115384> is submitted to queue <normal>.
测试 n=105，等待日志...
n=105 失败。
Job <115385> is submitted to queue <normal>.
测试 n=104，等待日志...
日志文件 /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g105-o.out 已存在，跳过作业 g105。
第一個失敗 n: 105
預計所有 >= 105 均失敗。


## 2. Global bgc test 

In [ ]:
import subprocess
import os
from joblib import Parallel, delayed
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed  # 切换到 ProcessPoolExecutor 以支持实时进度
import time
import sys

def load_environment(env_file):
    cmd = f'bash -c "source {env_file} && env"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    new_env = {}
    for line in result.stdout.strip().split('\n'):
        if '=' in line:
            key, value = line.split('=', 1)
            if not key.startswith('BASH_FUNC_'):
                new_env[key] = value
    
    os.environ.update(new_env)
    return os.environ.copy()


if __name__ == "__main__":
    env_file = '/share/home/dq089/soft/gnu-env'
    run_path = '/share/home/dq076/mode/ME/251004/run/'
    lsf_path = f'{run_path}g1.lsf'
    log_path = f'{run_path}logs/'  
    os.makedirs(log_path, exist_ok=True)

    updated_env = load_environment(env_file)

    for n in range(49,144):
        log_file = f'{log_path}g{n}-o.out'

        if os.path.exists(log_file):
            print(f"日志文件 {log_file} 已存在，跳过作业 g{n}。")
            continue

        with open(lsf_path, 'r') as file:
            lsf = file.readlines()

        for i, line in enumerate(lsf):
            if '#BSUB -J g' in line:
                lsf[i] = f'#BSUB -J g{n}\n'
            if '#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
                lsf[i] = f'#BSUB -o /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-o.out\n'
            if '#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/' in line:
                lsf[i] = f'#BSUB -e /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/logs/g{n}-e.out\n'
            if '#BSUB -n' in line:
                lsf[i] = f'#BSUB -n {n}\n'
            if '/share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n' in line:
                lsf[i] = f'mpirun -np {n} /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/mkinidata.x /share/home/dq076/mode/ME/CoLM202X_CH4_g/run/g1.nml\n'
        
        with open(lsf_path, 'w') as file:
            file.writelines(lsf)
        
        sub= f'bsub<{lsf_path}'
        os.system(sub)
        # subprocess.run(sub, 
        #                 env=updated_env, 
        #                 stderr=subprocess.STDOUT, 
        #                 text=True)
        
        print(f"提交作业 g{n}，等待日志文件 {log_file} 生成...")
        while not os.path.exists(log_file):
            time.sleep(1)  # 每秒检查一次

        # 文件存在后，检查内容
        with open(log_file, 'r', encoding='utf-8') as f:
            content = f.read()

        if "CoLM Initialization Execution Completed" in content:
            print(f"日志文件 {log_file} 已生成且包含完成消息，继续下一个作业。")
        else:
            print(f"日志文件 {log_file} 已生成，但未找到 'CoLM Initialization Execution Completed'，程序结束。")
            sys.exit(1)  # 或 raise ValueError("作业未完成") 以结束程序